In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, os, subprocess, sys
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXECUTION_EXACT='bcecbf63a2218eabd7dd878f19dd379feedf2b26'
REPO=Path('/content/cegwm-geometry-v6-r01')
DRIVE_RUNS=Path('/content/drive/MyDrive/CEG-WM/Geometry-V6/R01')
if REPO.exists(): raise FileExistsError('fresh runtime required')
subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
subprocess.run(['git','checkout','--detach',APPROVED_EXECUTION_EXACT],cwd=REPO,check=True)
def git(*args): return subprocess.run(['git',*args],cwd=REPO,check=True,capture_output=True,text=True).stdout.strip()
assert git('rev-parse','HEAD')==APPROVED_EXECUTION_EXACT and git('branch','--show-current')=='' and git('status','--porcelain')==''
RUN_ROOT=DRIVE_RUNS/f'{APPROVED_EXECUTION_EXACT}-{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}'
if RUN_ROOT.exists(): raise FileExistsError('create-only R0.1 run exists')
RUN_ROOT.mkdir(parents=True,exist_ok=False)


In [ ]:
from getpass import getpass
assert __import__('torch').cuda.is_available(), 'GPU required; no R0.1 record on CPU'
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
content_key=os.environ.get('CEG_WM_ROOT_KEY') or getpass('CEG_WM_ROOT_KEY: ')
hf_token=os.environ.get('HF_TOKEN') or getpass('HF_TOKEN: ')
assert all(isinstance(value,str) and value.strip() for value in (content_key,hf_token))
markers=('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL')
runner_env={key:value for key,value in os.environ.items() if not any(marker in key.upper() for marker in markers)}
runner_env.update({'CEG_WM_ROOT_KEY':content_key,'HF_TOKEN':hf_token})
command=[sys.executable,'-m','experiments.geometry_v6_r01_engine','--run-diagnostic','--repo-root',str(REPO),'--expected-exact',APPROVED_EXECUTION_EXACT,'--output-json',str(RUN_ROOT/'r01.json')]
def scrub(value, *secrets):
    for secret in secrets: value=value.replace(secret,'[REDACTED]')
    return value[-2000:]
try:
    completed=subprocess.run(command,cwd=REPO,env=runner_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,check=False)
    summary=scrub(completed.stdout,content_key,hf_token); failure=scrub(completed.stderr,content_key,hf_token)
finally:
    content_key=hf_token=''
    runner_env.clear()
print(summary)
if completed.returncode!=0:
    print(failure)
    raise RuntimeError('Geometry-V6 R0.1 stopped; retained create-only path: '+str(RUN_ROOT))
artifact=RUN_ROOT/'r01.json'
print({'path':str(artifact),'sha256':hashlib.sha256(artifact.read_bytes()).hexdigest(),'exact':APPROVED_EXECUTION_EXACT})
